In [ ]:
# Install required packages
!pip install sentencepiece

# Download the Mistral-7B-v0.1 model tar file
!wget https://files.mistral-7b-v0-1.mistral.ai/mistral-7B-v0.1.tar

# Extract the downloaded tar file
!tar -xf mistral-7B-v0.1.tar

In [ ]:
# Import libraries
from pathlib import Path
import json
import math
from dataclasses import dataclass
import torch
from torch import nn
from sentencepiece import SentencePieceProcessor

# 1) Mistral LLM
[Mistral Github](https://github.com/mistralai/mistral-src/blob/main/one_file_ref.py)

TODO: Explore original cache, prefill chunking (curently using one file ref)

## 1.1 Config

In [ ]:
# Define model hyperparameter configuration dataclass
@dataclass
class ModelArgs:
    dim: int            # embed_dim / model_dim
    n_layers: int       # num_layers
    head_dim: int       # head_dim
    hidden_dim: int     # hidden_dim (FFN intermediate size)
    n_heads: int        # num_heads
    n_kv_heads: int     # number of key/value heads (GQA)
    sliding_window: int # sliding window size for local attention
    norm_eps: float
    vocab_size: int     # vocab_size
    max_batch_size: int = 0  # batch_num

# Define the path to the model
model_path = './mistral-7B-v0.1'

# Define a sample dataset
data_set = [
    "Lucky is a dog from my neighbour.",
    "She likes to play ball with Lucy.",
    "Lucy is her best friend.",
]

# Define the number of pipeline ranks
num_pipeline_ranks = 2

# if num_pipeline_ranks > 1:
#     torch.distributed.init_process_group()
#     torch.cuda.set_device(torch.distributed.get_rank())
#     should_print = torch.distributed.get_rank() == 0

## 1.2 Tokenization

In [ ]:
# Define a Tokenizer class wrapping SentencePiece
class Tokenizer:
    def __init__(self, model_path: str):
        # Assert that the model path exists
        assert Path(model_path).exists(), model_path
        # Initialize SentencePieceProcessor
        self._model = SentencePieceProcessor(model_file=model_path)
        # Assert that vocab_size matches piece size
        assert self._model.vocab_size() == self._model.get_piece_size()

    @property
    def n_words(self):
        # Return vocab_size
        return self._model.vocab_size()

    @property
    def bos_id(self) -> int:
        # Return the beginning-of-sequence token ID
        return self._model.bos_id()

    @property
    def eos_id(self):
        # Return the end-of-sequence token ID
        return self._model.eos_id()

    @property
    def pad_id(self):
        # Return the padding token ID
        return self._model.pad_id()

    def encode(self, s: str, bos: bool=True):
        # Encode a string into token IDs → Output: List[int] of length seq_len
        t = self._model.encode(s)
        if bos:
            t = [self.bos_id, *t]
        return t

    def decode(self, t):
        # Decode a list of token IDs back into a string
        return self._model.decode(t)

In [ ]:
# Instantiate tokenizer and encode all prompts
tokenizer = Tokenizer(f'{model_path}/tokenizer.model')
# encoded_prompts: List of lists, each inner list shape: (seq_len_i,)
encoded_prompts = [tokenizer.encode(prompt) for prompt in data_set]
print(encoded_prompts)

# Compute per-prompt lengths
prompt_lens = [len(x) for x in encoded_prompts]
min_prompt_len = min(prompt_lens)
max_prompt_len = max(prompt_lens)
print('Length: ', min_prompt_len, max_prompt_len)

# Create padded input tensor filled with pad_id
# Input: (batch_num, max_prompt_len) — initialized to pad_id
input_tokens = torch.full(
    (len(data_set), max_prompt_len),
    tokenizer.pad_id,
    dtype=torch.long,
    device="cuda"
)
print(f"Empty Tensor: {input_tokens}")

# Fill in actual token IDs; positions beyond prompt length remain pad_id
# input_tokens shape: (batch_num, max_prompt_len)
for i, encoded in enumerate(encoded_prompts):
    input_tokens[i, :len(encoded)] = torch.tensor(encoded).to(input_tokens)
print(f"Tokenized Tensor: {input_tokens}")

# Create a boolean mask identifying real (non-pad) token positions
# input_mask shape: (batch_num, max_prompt_len)
input_mask = input_tokens != tokenizer.pad_id

# Create 1-D position indices for the prefix up to min_prompt_len
# positions shape: (min_prompt_len,)
positions = torch.arange(0, min_prompt_len).to("cuda")
print(f"Position: {positions}")

[[1, 393, 11791, 349, 264, 3914, 477, 586, 18583, 28723], [1, 985, 12672, 298, 1156, 4374, 395, 18010, 28723], [1, 18010, 349, 559, 1489, 1832, 28723]]
Length:  7 10
Empty Tensor: tensor([[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1]], device='cuda:0')
Tokenized Tensor: tensor([[    1,   393, 11791,   349,   264,  3914,   477,   586, 18583, 28723],
        [    1,   985, 12672,   298,  1156,  4374,   395, 18010, 28723,    -1],
        [    1, 18010,   349,   559,  1489,  1832, 28723,    -1,    -1,    -1]],
       device='cuda:0')
Position: tensor([0, 1, 2, 3, 4, 5, 6], device='cuda:0')


## 1.3 Model

In [ ]:
# Instantiate ModelArgs with small demo dimensions
# batch_num=3, seq_len variable, embed_dim=512, num_heads=4, head_dim=128,
# hidden_dim=2048, num_layers=1, vocab_size=32_000
args = ModelArgs(
    dim=512,
    n_layers=1,
    head_dim=128,
    hidden_dim=2048,
    n_heads=4,
    n_kv_heads=2,
    sliding_window=3,
    norm_eps=1e-5,
    vocab_size=32_000,
    max_batch_size=3,
)

# Official 7B parameter configuration (for reference):
# {
#     "dim": 4096,          # embed_dim
#     "n_layers": 32,       # num_layers
#     "head_dim": 128,      # head_dim
#     "hidden_dim": 14336,  # hidden_dim
#     "n_heads": 32,        # num_heads
#     "n_kv_heads": 8,
#     "norm_eps": 1e-05,
#     "sliding_window": 4096,
#     "vocab_size": 32000   # vocab_size
# }

In [ ]:
# Utility: reshape freqs_cis tensor for broadcasting with query/key tensors
def _reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    # Get the number of dimensions of the input tensor
    ndim = x.ndim

    # Build broadcast shape: keep dim 1 (seq_len) and last dim (head_dim//2), set others to 1
    # Output shape: (1, seq_len, 1, head_dim // 2)
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]

    # Input: (seq_len, head_dim // 2) -> Output: (1, seq_len, 1, head_dim // 2)
    return freqs_cis.view(*shape)


# Apply Rotary Position Embeddings (RoPE) to query and key tensors
def apply_rotary_emb(xq, xk, freqs_cis):

    # Cast to complex: pair last-dim elements as (real, imag)
    # Input xq:  (batch_num, seq_len, num_heads,    head_dim)
    # Output xq_: (batch_num, seq_len, num_heads,    head_dim // 2)  complex
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))

    # Input xk:  (batch_num, seq_len, num_kv_heads, head_dim)
    # Output xk_: (batch_num, seq_len, num_kv_heads, head_dim // 2)  complex
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))

    # Reshape freqs_cis for broadcasting
    # Input: (seq_len, head_dim // 2) -> Output: (1, seq_len, 1, head_dim // 2)
    freqs_cis = _reshape_for_broadcast(freqs_cis, xq_)

    # Rotate query by multiplying with complex frequencies
    # Input xq_:   (batch_num, seq_len, num_heads,    head_dim // 2) complex
    # Output xq_out: (batch_num, seq_len, num_heads,  head_dim // 2, 2) real
    xq_out = torch.view_as_real(xq_ * freqs_cis)

    # Rotate key by multiplying with complex frequencies
    # Input xk_:   (batch_num, seq_len, num_kv_heads, head_dim // 2) complex
    # Output xk_out: (batch_num, seq_len, num_kv_heads, head_dim // 2, 2) real
    xk_out = torch.view_as_real(xk_ * freqs_cis)

    # Flatten last two dims back to head_dim
    # Input: (batch_num, seq_len, num_heads,    head_dim // 2, 2) -> Output: (batch_num, seq_len, num_heads,    head_dim)
    xq_out = xq_out.reshape(*xq.shape)
    # Input: (batch_num, seq_len, num_kv_heads, head_dim // 2, 2) -> Output: (batch_num, seq_len, num_kv_heads, head_dim)
    xk_out = xk_out.reshape(*xk.shape)

    return xq_out.type_as(xq), xk_out.type_as(xk)


# Expand KV heads to match query head count (Grouped Query Attention)
def repeat_kv(keys: torch.Tensor, values: torch.Tensor, repeats: int):
    """
    Part of Grouped Query Attention.
    Repeat heads of key and values to match the dimension of query.
    """
    # Input keys:   (batch_num, seq_len, num_kv_heads, head_dim)
    # Output keys:  (batch_num, seq_len, num_kv_heads * repeats, head_dim)  i.e. num_heads
    keys = torch.repeat_interleave(keys, repeats=repeats, dim=2)

    # Input values:  (batch_num, seq_len, num_kv_heads, head_dim)
    # Output values: (batch_num, seq_len, num_kv_heads * repeats, head_dim)  i.e. num_heads
    values = torch.repeat_interleave(values, repeats=repeats, dim=2)
    return keys, values

**Sliding window attention**

![](https://miro.medium.com/v2/resize:fit:1400/0*uJ9qfE3Ik92XnEdz)

Information Flow from Input to Upper Layer.
Note that tokens outside the sliding window still influence next word prediction. At each attention layer, information can move forward by W tokens at most: after two attention layers, information can move forward by 2W tokens, etc.

For instance in a sequence of length 16K and a sliding window of 4K, after 4 layers, information has propagated to the full sequence length.

**Rolling buffer cache**

![](https://github.com/mistralai/mistral-src/raw/main/assets/rolling_cache.png)

The cache has a fixed size of W, and we store the (key, value) for position i in cache position i % W. When the position i is larger than W, past values in the cache are overwritten.

In [ ]:
# Define Attention with Sliding Window Attention and Grouped Query Attention (GQA)
class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args

        # Store head/attention configuration
        self.n_heads = args.n_heads        # num_heads
        self.n_kv_heads = args.n_kv_heads  # GQA KV heads
        self.repeats = self.n_heads // self.n_kv_heads
        self.sliding_window = self.args.sliding_window
        self.scale = self.args.head_dim**-0.5  # 1/sqrt(head_dim)

        # Project input embed_dim to Q/K/V spaces
        # wq: embed_dim -> num_heads * head_dim
        self.wq = nn.Linear(args.dim, args.n_heads * args.head_dim, bias=False)
        # wk: embed_dim -> num_kv_heads * head_dim
        self.wk = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
        # wv: embed_dim -> num_kv_heads * head_dim
        self.wv = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
        # wo: num_heads * head_dim -> embed_dim
        self.wo = nn.Linear(args.n_heads * args.head_dim, args.dim, bias=False)

        # Allocate rolling KV cache for sliding window
        # cache_k shape: (batch_num, sliding_window, num_kv_heads, head_dim)
        self.cache_k = torch.empty(
            (
                args.max_batch_size,
                args.sliding_window,
                self.n_kv_heads,
                self.args.head_dim,
            ), dtype=torch.float32).cuda()

        # cache_v shape: (batch_num, sliding_window, num_kv_heads, head_dim)
        self.cache_v = torch.empty(
            (
                args.max_batch_size,
                args.sliding_window,
                self.n_kv_heads,
                self.args.head_dim,
            ), dtype=torch.float32).cuda()

    def forward(self, x, freqs_cis, positions, mask):
        """
        x:         (batch_num, seq_len, embed_dim)
        freqs_cis: (seq_len, head_dim // 2)
        positions: (seq_len,)
        mask:      (seq_len, seq_len) causal + sliding-window mask
        """

        # Unpack batch and sequence dimensions
        bsz, seqlen, _ = x.shape

        # Project input to Q, K, V
        # Input x: (batch_num, seq_len, embed_dim)
        # Output xq: (batch_num, seq_len, num_heads * head_dim)
        # Output xk: (batch_num, seq_len, num_kv_heads * head_dim)
        # Output xv: (batch_num, seq_len, num_kv_heads * head_dim)
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        ### grouped query attention ###
        # Reshape to separate heads dimension
        # Input: (batch_num, seq_len, num_heads * head_dim) -> Output xq: (batch_num, seq_len, num_heads, head_dim)
        xq = xq.view(bsz, seqlen, self.n_heads, self.args.head_dim)
        # Input: (batch_num, seq_len, num_kv_heads * head_dim) -> Output xk: (batch_num, seq_len, num_kv_heads, head_dim)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.args.head_dim)
        # Input: (batch_num, seq_len, num_kv_heads * head_dim) -> Output xv: (batch_num, seq_len, num_kv_heads, head_dim)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.args.head_dim)

        # Apply Rotary Position Embeddings to Q and K
        # Input xq: (batch_num, seq_len, num_heads,    head_dim) -> Output: same shape, rotated
        # Input xk: (batch_num, seq_len, num_kv_heads, head_dim) -> Output: same shape, rotated
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)

        # Compute rolling-buffer write positions: positions mod sliding_window
        # [0, 1, 2, 3, 4, 5, 6] --> [0, 1, 2, 0, 1, 2, 0] --> [1, 2, 0] --> (1, sliding_window, 1, 1)
        # scatter_pos before repeat: (1, min(seq_len, sliding_window), 1, 1)
        scatter_pos = (positions[-self.sliding_window:] % self.sliding_window)[None, :, None, None]
        # Expand scatter_pos to match cache shape
        # scatter_pos shape: (batch_num, sliding_window, num_kv_heads, head_dim)
        scatter_pos = scatter_pos.repeat(bsz, 1, self.n_kv_heads, self.args.head_dim)

        # Write current K/V into the rolling sliding-window KV cache
        # cache_k[:bsz] shape: (batch_num, sliding_window, num_kv_heads, head_dim)
        # it rotates according to the index: ['on', 'the', 'cat', 'is'] -> ['the', 'cat', 'is', 'on']
        self.cache_k[:bsz].scatter_(dim=1, index=scatter_pos, src=xk[:, -self.sliding_window:])
        self.cache_v[:bsz].scatter_(dim=1, index=scatter_pos, src=xv[:, -self.sliding_window:])

        # Expand KV heads to match num_heads (GQA repeat)
        if positions.shape[0] > 1:
            # Prefill path: use current sequence K/V
            # Input: (batch_num, seq_len, num_kv_heads, head_dim) -> Output: (batch_num, seq_len, num_heads, head_dim)
            key, value = repeat_kv(xk, xv, self.repeats)
        # use cache if seq_len is 1 (decode step)
        else:
            # Decode path: retrieve from KV cache up to cur_pos
            cur_pos = positions[-1].item() + 1
            # Input: (batch_num, cur_pos, num_kv_heads, head_dim) -> Output: (batch_num, cur_pos, num_heads, head_dim)
            key, value = repeat_kv(
                self.cache_k[:bsz, :cur_pos, ...], self.cache_v[:bsz, :cur_pos, ...], self.repeats
            )

        ### original ###
        # xformers requires (B=1, S, H, D)
        # xq, key, val = xq[None, ...], key[None, ...], val[None, ...]
        # output = memory_efficient_attention(
        #     xq, key, val, None if cache is None else cache.mask
        # )
        # return self.wo(output.view(seqlen_sum, self.n_heads * self.head_dim))

        # Transpose heads to front for batched matmul
        # Input: (batch_num, seq_len, num_heads, head_dim) -> Output query: (batch_num, num_heads, seq_len, head_dim)
        query = xq.transpose(1, 2)
        # Input: (batch_num, seq_len, num_heads, head_dim) -> Output key: (batch_num, num_heads, seq_len, head_dim)
        key = key.transpose(1, 2)
        # Input: (batch_num, seq_len, num_heads, head_dim) -> Output value: (batch_num, num_heads, seq_len, head_dim)
        value = value.transpose(1, 2)

        # Scaled dot-product attention scores
        # Q: (batch_num, num_heads, seq_len, head_dim) x K^T: (batch_num, num_heads, head_dim, seq_len)
        # Output scores: (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(query, key.transpose(2, 3)) * self.scale

        # Apply sliding-window causal mask (additive log mask, -inf for masked positions)
        # mask broadcast shape: (batch_num, num_heads, seq_len, seq_len)
        if mask is not None:
            scores += mask[None, None, ...]
        scores = scores.float()

        # Softmax over key dimension to get attention weights
        # Input: (batch_num, num_heads, seq_len, seq_len) -> Output: same shape, sum-to-1 along last dim
        scores = nn.functional.softmax(scores, dim=-1).type_as(query)

        # Weighted sum of value vectors
        # Input scores: (batch_num, num_heads, seq_len, seq_len), value: (batch_num, num_heads, seq_len, head_dim)
        # Output: (batch_num, num_heads, seq_len, head_dim)
        output = torch.matmul(scores, value)

        # Merge heads and project back to embed_dim
        # Input: (batch_num, num_heads, seq_len, head_dim) -> (batch_num, seq_len, num_heads * head_dim) -> (batch_num, seq_len, embed_dim)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        # Input: (batch_num, seq_len, num_heads * head_dim) -> Output: (batch_num, seq_len, embed_dim)
        return self.wo(output)

In [ ]:
# Define the SwiGLU FeedForward network (used in Mistral)
class FeedForward(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        # w1: gate projection  embed_dim -> hidden_dim
        self.w1 = nn.Linear(args.dim, args.hidden_dim, bias=False)
        # w2: down projection  hidden_dim -> embed_dim
        self.w2 = nn.Linear(args.hidden_dim, args.dim, bias=False)
        # w3: up projection    embed_dim -> hidden_dim
        self.w3 = nn.Linear(args.dim, args.hidden_dim, bias=False)

    def forward(self, x) -> torch.Tensor:
        # SwiGLU activation: silu(w1(x)) * w3(x), then project down
        # Input x: (batch_num, seq_len, embed_dim)
        # w1(x): (batch_num, seq_len, hidden_dim)
        # w3(x): (batch_num, seq_len, hidden_dim)
        # silu(w1(x)) * w3(x): (batch_num, seq_len, hidden_dim)  <- element-wise gate
        # Output: (batch_num, seq_len, embed_dim)
        return self.w2(nn.functional.silu(self.w1(x)) * self.w3(x))

In [ ]:
# Define RMSNorm normalization layer
class RMSNorm(torch.nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Learnable per-dimension scale parameter; shape: (embed_dim,)
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        # Compute RMS normalization (no mean subtraction)
        # Input: (batch_num, seq_len, embed_dim) -> Output: (batch_num, seq_len, embed_dim)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # Normalize then scale by learned weight
        # Input: (batch_num, seq_len, embed_dim) -> Output: (batch_num, seq_len, embed_dim)
        output = self._norm(x.float()).type_as(x)
        # weight shape: (embed_dim,) broadcast over batch and seq dims
        return output * self.weight


# Define a single Transformer Block (Attention + FeedForward with residual connections)
class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        # Get number of heads and embedding dimension
        self.n_heads = args.n_heads
        self.dim = args.dim
        # Sliding-window multi-head attention
        self.attention = Attention(args)
        # SwiGLU FeedForward
        self.feed_forward = FeedForward(args=args)
        # Pre-attention RMSNorm
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        # Pre-FFN RMSNorm
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.args = args

    def forward(self, x, freqs_cis, positions, mask):

        # Pre-norm before attention
        # Input x: (batch_num, seq_len, embed_dim) -> norm_x: (batch_num, seq_len, embed_dim)
        norm_x = self.attention_norm(x)

        # Sliding-window self-attention
        # Input: (batch_num, seq_len, embed_dim) -> Output r: (batch_num, seq_len, embed_dim)
        r = self.attention.forward(norm_x, freqs_cis, positions, mask)

        # First residual connection
        # h shape: (batch_num, seq_len, embed_dim)
        h = x + r

        # Pre-norm before FFN
        # Input h: (batch_num, seq_len, embed_dim) -> norm_h: (batch_num, seq_len, embed_dim)
        norm_h = self.ffn_norm(h)

        # SwiGLU FeedForward
        # Input: (batch_num, seq_len, embed_dim) -> Output r: (batch_num, seq_len, embed_dim)
        r = self.feed_forward.forward(norm_h)

        # Second residual connection
        # out shape: (batch_num, seq_len, embed_dim)
        out = h + r
        return out

In [ ]:
# Precompute complex-valued rotation frequencies for RoPE
def precompute_freqs_cis(head_dim, end=128_000, theta=10000.0):
    # Compute base frequencies: 1 / theta^(2i/head_dim) for i in 0..head_dim//2
    # freqs shape: (head_dim // 2,)
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))

    # Time steps (one per sequence position up to end = max_seq_len)
    # t shape: (end,)
    t = torch.arange(end, device=freqs.device)

    # Outer product: each position gets a scaled frequency vector
    # Input t: (end,), freqs: (head_dim // 2,) -> Output freqs: (end, head_dim // 2)
    freqs = torch.outer(t, freqs).float()

    # Convert to unit complex numbers e^(i*freq) for rotation
    # Input: (end, head_dim // 2) -> Output: (end, head_dim // 2) complex
    return torch.polar(torch.ones_like(freqs), freqs)


# Define the full Mistral Transformer model
class Transformer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.vocab_size = args.vocab_size   # vocab_size
        self.n_layers = args.n_layers        # num_layers

        # Token embedding table: vocab_size -> embed_dim
        # tok_embeddings weight shape: (vocab_size, embed_dim)
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)

        # Stack of num_layers TransformerBlocks
        self.layers = torch.nn.ModuleList(
            [TransformerBlock(args=args) for _ in range(self.n_layers)]
        )

        # Final RMSNorm before output projection
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)

        # Output projection: embed_dim -> vocab_size (language model head)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

        # Precompute and cache RoPE frequencies
        # freqs_cis shape: (128_000, head_dim // 2) complex
        self.freqs_cis = precompute_freqs_cis(self.args.head_dim, 128_000).to("cuda")

    def forward(self, input_ids, positions):
        # input_ids shape: (batch_num, seq_len)
        # positions shape: (seq_len,)

        # Embed input tokens
        # Input: (batch_num, seq_len) -> Output h: (batch_num, seq_len, embed_dim)
        h = self.tok_embeddings(input_ids)

        # Slice precomputed RoPE frequencies for current positions
        # freqs_cis shape: (seq_len, head_dim // 2)
        freqs_cis = self.freqs_cis[positions]

        mask = None
        # Build causal + sliding-window attention mask for prefill (seq_len > 1)
        if input_ids.shape[1] > 1:
            seqlen = input_ids.shape[1]
            # Initialize (seq_len, seq_len) matrix of ones
            # tensor shape: (seq_len, seq_len)
            tensor = torch.full(
                (seqlen, seqlen),
                dtype=h.dtype,
                fill_value=1,
                device=h.device,
            )

            # Lower-triangular causal mask
            # Input: (seq_len, seq_len) -> Output mask: (seq_len, seq_len)
            mask = torch.tril(tensor, diagonal=0).to(h.dtype)

            # Apply sliding window: keep only the last sliding_window positions
            # Input: (seq_len, seq_len) -> Output mask: (seq_len, seq_len)
            mask = torch.triu(mask, diagonal=-self.args.sliding_window)

            # Convert to log scale: 0 -> 0.0, masked -> -inf (additive to attention scores)
            # Input: (seq_len, seq_len) -> Output mask: (seq_len, seq_len)
            mask = torch.log(mask)

        # Pass through all num_layers TransformerBlocks
        # Input h: (batch_num, seq_len, embed_dim) -> Output h: (batch_num, seq_len, embed_dim)
        for layer in self.layers:
            # h shape: (batch_num, seq_len, embed_dim)
            h = layer(h, freqs_cis, positions, mask)

        # Apply final norm then project to vocabulary logits
        # norm(h): (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, vocab_size)
        return self.output(self.norm(h)).float()

    ### Load from pretrained checkpoint ###
    @staticmethod
    def from_folder(folder, max_batch_size=1, device="cuda", dtype=torch.float16):
        # Load model arguments from params.json
        with open(f'{folder}/params.json', 'r') as f:
            model_args = ModelArgs(**json.loads(f.read()))
        # Set max batch size
        model_args.max_batch_size = max_batch_size
        # Instantiate model on target device
        model = Transformer(model_args).to(device=device, dtype=dtype)
        # Load pretrained weights
        loaded = torch.load(f'{folder}/consolidated.00.pth')
        model.load_state_dict(loaded)
        return model

In [ ]:
# Instantiate Mistral Transformer and run a test forward pass
# Input: (batch_num=3, min_prompt_len)
model = Transformer(args).to("cuda", dtype=torch.float32)

# Forward pass -> logits over vocabulary
# Input: (batch_num, min_prompt_len) -> Output logits: (batch_num, min_prompt_len, vocab_size)
logits = model.forward(input_tokens[:, :min_prompt_len], positions)

# Compute log-probabilities via log-softmax over vocab dimension
# Input: (batch_num, min_prompt_len, vocab_size) -> Output logprobs: (batch_num, min_prompt_len, vocab_size)
logprobs = nn.functional.log_softmax(logits, dim=-1)

print(logits.size(), logprobs.size())

torch.Size([3, 7, 32000]) torch.Size([3, 7, 32000])


## 1.4 Generation

In [ ]:
# (Optional) Load pretrained model from disk -- commented out for demo
# model = model.from_folder("./mistral-7B-v0.1")
# model.eval()

# Define the model path
model_path = './mistral-7B-v0.1'

# Define generation prompts
prompts = [
    "Lucky is a dog from my neighbour.",
    "She likes to play ball with Lucy.",
    "Lucy is her best friend.",
]

# Instantiate tokenizer and encode all prompts into token ID lists
tokenizer = Tokenizer(f'{model_path}/tokenizer.model')
# encoded_prompts: List[List[int]], each list has shape (seq_len_i,)
encoded_prompts = [tokenizer.encode(prompt) for prompt in prompts]

In [ ]:
# Compute minimum and maximum prompt lengths across the batch
min_prompt_len = min(prompt_lens)
max_prompt_len = max(prompt_lens)

# Allocate padded input tensor of shape (batch_num, max_prompt_len)
# Input: (batch_num, max_prompt_len) -- filled with pad_id
input_tokens = torch.full(
    (len(prompts), max_prompt_len),
    tokenizer.pad_id,
    dtype=torch.long,
    device="cuda"
)

# Fill in actual token IDs for each prompt
# input_tokens shape: (batch_num, max_prompt_len)
for i, encoded in enumerate(encoded_prompts):
    input_tokens[i, :len(encoded)] = torch.tensor(encoded).to(input_tokens)

# Boolean mask: True where token is real, False where padded
# input_mask shape: (batch_num, max_prompt_len)
input_mask = input_tokens != tokenizer.pad_id
print(input_tokens)

tensor([[    1,   393, 11791,   349,   264,  3914,   477,   586, 18583, 28723,
            -1],
        [    1,   985, 12672,   298,  1156,  4374,   395, 18010, 28723,    -1,
            -1],
        [    1, 18010,   349,   559,  1489,  1832, 28723,    -1,    -1,    -1,
            -1]], device='cuda:0')


In [ ]:
# Create position indices for the shared prefix
# positions shape: (min_prompt_len,)
positions = torch.arange(0, min_prompt_len).to("cuda")

# Prefill forward pass -- process the shared prompt prefix
# Input: (batch_num, min_prompt_len) -> Output logits: (batch_num, min_prompt_len, vocab_size)
logits = model.forward(input_tokens[:, :min_prompt_len], positions)

# Compute log-probabilities over the vocabulary
# Input: (batch_num, min_prompt_len, vocab_size) -> Output logprobs: (batch_num, min_prompt_len, vocab_size)
logprobs = nn.functional.log_softmax(logits, dim=-1)
print(logprobs.size())

# Gather log-probabilities for each actual next token in the prompt
# logprobs[:, :-1, :]:  (batch_num, min_prompt_len - 1, vocab_size)
# input_tokens[:, 1:min_prompt_len, None]: index tensor (batch_num, min_prompt_len - 1, 1)
# Output: (batch_num, min_prompt_len - 1)
all_logprobs = [
    logprobs[:, :-1, :].gather(2, input_tokens[:, 1:min_prompt_len, None]).squeeze(-1),
]
print(all_logprobs)

torch.Size([3, 7, 32000])
[tensor([[-11.2580, -10.4556, -10.4530, -11.7142,  -9.8308, -10.6482],
        [-11.0530, -10.4912, -10.6849, -10.0249, -10.4921, -11.3246],
        [-10.3404, -11.1509, -10.4690, -11.3796,  -9.9112, -11.8490]],
       device='cuda:0', grad_fn=<SqueezeBackward1>)]


In [ ]:
# Top-p (nucleus) sampling helper
def sample_top_p(probs: torch.Tensor, p: float):
    # probs shape: (batch_num, vocab_size)
    assert 0 <= p <= 1

    # Sort probabilities in descending order
    # probs_sort shape: (batch_num, vocab_size)
    # probs_idx shape:  (batch_num, vocab_size)  -- original indices
    probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)

    # Cumulative sum of sorted probabilities
    # probs_sum shape: (batch_num, vocab_size)
    probs_sum = torch.cumsum(probs_sort, dim=-1)

    # Build mask: exclude tokens once cumulative probability exceeds p
    # mask shape: (batch_num, vocab_size)
    mask = probs_sum - probs_sort > p

    # Zero out excluded token probabilities
    probs_sort[mask] = 0.0

    # Re-normalize remaining probabilities to sum to 1
    # probs_sort shape: (batch_num, vocab_size)
    probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))

    # Sample one token per batch element from the nucleus distribution
    # Input: (batch_num, vocab_size) -> Output next_token: (batch_num, 1)
    next_token = torch.multinomial(probs_sort, num_samples=1)

    # torch gather is indexing
    # Map sampled index back to original vocabulary index
    # Output: (batch_num, 1)
    return torch.gather(probs_idx, -1, next_token)


# Sampling dispatcher supporting temperature scaling and top-p
def sample(logits: torch.Tensor, temperature: float, top_p: float):

    # logits size: (batch_num, vocab_size)
    if temperature > 0:
        # Apply temperature and convert to probabilities
        # Input: (batch_num, vocab_size) -> Output probs: (batch_num, vocab_size)
        probs = torch.softmax(logits / temperature, dim=-1)

        # Sample via top-p nucleus sampling
        # Input: (batch_num, vocab_size) -> Output next_token: (batch_num, 1)
        next_token = sample_top_p(probs, top_p)
    else:
        # Greedy decoding: take argmax over vocabulary
        # Input: (batch_num, vocab_size) -> Output next_token: (1, batch_num) then reshaped
        next_token = torch.argmax(logits, dim=-1).unsqueeze(0)

    # Flatten to (batch_num,)
    return next_token.reshape(-1)

In [ ]:
# Autoregressive token generation loop
max_tokens = 10
# Sampling temperature
temperature = 0.8
# Accumulator for generated token tensors
generated = []

# start from the min_prompt_len (for batch >1)
# loop until getting the max tokens

# Iterate to generate one token per step
for cur_pos in range(min_prompt_len, max_tokens):

    # Sample next token from last position log-probs
    # logprobs[:, -1, :] shape: (batch_num, vocab_size)
    # next_token shape: (batch_num,)
    next_token = sample(logprobs[:, -1, :], temperature=temperature, top_p=0.8)

    # For positions within the input, use ground-truth token if not padded
    if cur_pos < input_mask.shape[1]:
        # if not pad, return original token, else return predicted token
        # Conditionally override with ground-truth token
        # next_token shape: (batch_num,)
        next_token = torch.where(
            input_mask[:, cur_pos],
            input_tokens[:, cur_pos],
            next_token
        )

    # Collect per-step log-probability for the chosen token
    # logprobs[:, -1, :].gather(1, next_token[:, None]) shape: (batch_num, 1)
    all_logprobs.append(logprobs[:, -1, :].gather(1, next_token[:, None]))

    # (max_tokens - min_prompt_len, batch_num, 1)
    # Store generated token
    # next_token[:, None] shape: (batch_num, 1)
    generated.append(next_token[:, None])

    # Decode step: feed single new token to get next prediction
    # Input: (batch_num, 1) -> Output logits: (batch_num, 1, vocab_size)
    logits = model.forward(next_token[:, None], torch.LongTensor([cur_pos]).to(next_token))

    # Compute log-probabilities for next iteration
    # Input: (batch_num, 1, vocab_size) -> Output logprobs: (batch_num, 1, vocab_size)
    logprobs = nn.functional.log_softmax(logits, dim=-1)

# Concatenate per-step log-probs into a single tensor
# all_logprobs shape: (batch_num, max_tokens)
all_logprobs = torch.cat(all_logprobs, 1)
print('All Prob Size: ', all_logprobs.size())

# Concatenate all generated token IDs
# generated shape: (batch_num, max_tokens - min_prompt_len)
generated = torch.cat(generated, 1)
print('Generated Tokens: ', generated)

# Decode generated token sequences back to text strings
res = []
for i, x in enumerate(encoded_prompts):
    res.append(tokenizer.decode(x[:min_prompt_len] + generated[i].tolist()))

for x in res:
    print(x)
    print("=====================")

All Prob Size:  torch.Size([3, 16])
Generated Tokens:  tensor([[  586, 18583, 28723, 29919, 31342,  6654,  4228,   399, 31841, 15854],
        [18010, 28723,   948, 16018, 28367,   689, 15099, 20251,  3445,  6302],
        [30021, 30327, 17565, 17159,  1603,  3185, 29532, 24422,  4040, 10668]],
       device='cuda:0')
Lucky is a dog from my neighbour.费ইrior School R❒ gri
She likes to play ball with Lucy. end dés (+ Chistes fucked command college
Lucy is her best friend.技颜 ALL cultiv////CON效itungbig Wild


# 2) Mixtral Mix of Expert (8x7b MoE)

1. Ensemble technique with multiple expert (e.g. some experts specialized for different languages or tasks).  
2. The output of experts are combined (weighted sum or averaging)
3. Only 2 out of 8 experts are used for each token
4. Before going into experts network, the gate produces logits to select topK experts.

[MoE One File Ref](https://github.com/mistralai/mistral-src/blob/main/moe_one_file_ref.py)


In [ ]:
# ── Section 2: Mixtral 8x7B Mixture-of-Experts ──────────────────────────────

# Dataclass holding MoE routing hyperparameters
@dataclass
class MoeArgs():
    num_experts: int          # total number of expert FFNs
    num_experts_per_tok: int  # top-k experts activated per token

# Extended ModelArgs for MoE model (no sliding window)
@dataclass
class ModelArgs():
    dim: int            # embed_dim
    n_layers: int       # num_layers
    head_dim: int       # head_dim
    hidden_dim: int     # hidden_dim (per-expert FFN intermediate size)
    n_heads: int        # num_heads
    n_kv_heads: int     # GQA KV heads
    norm_eps: float
    vocab_size: int     # vocab_size
    moe: MoeArgs        # MoE routing config
    max_batch_size: int = 0
    max_seq_len: int = 0

# Instantiate small demo MoE config: 4 experts, top-2 routing
moe_args = MoeArgs(num_experts=4, num_experts_per_tok=2)

# Instantiate model args with MoE config
# batch_num=3, embed_dim=512, num_heads=4, head_dim=128, hidden_dim=2048, num_layers=1
args = ModelArgs(
    dim=512,
    n_layers=1,
    head_dim=128,
    hidden_dim=2048,
    n_heads=4,
    n_kv_heads=2,
    norm_eps=1e-5,
    vocab_size=32_000,
    max_batch_size=3,
    max_seq_len=20,
    moe=moe_args
)

In [ ]:
# Define the Attention module for Mixtral (full-context GQA, no sliding window, uses KV cache)
class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args

        # Store head/attention configuration
        self.n_heads: int = args.n_heads        # num_heads
        self.n_kv_heads: int = args.n_kv_heads  # GQA KV heads

        self.repeats = self.n_heads // self.n_kv_heads
        self.scale = self.args.head_dim**-0.5

        # Linear projections
        # wq: embed_dim -> num_heads * head_dim
        self.wq = nn.Linear(args.dim, args.n_heads * args.head_dim, bias=False)
        # wk: embed_dim -> num_kv_heads * head_dim
        self.wk = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
        # wv: embed_dim -> num_kv_heads * head_dim
        self.wv = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
        # wo: num_heads * head_dim -> embed_dim
        self.wo = nn.Linear(args.n_heads * args.head_dim, args.dim, bias=False)

        # Lazy-initialized KV cache (allocated on first forward call)
        self._cache_k = None
        self._cache_v = None

    # Lazily allocate KV cache tensors on the same device/dtype as input
    def get_caches(self, x: torch.Tensor):
        dtype, device = x.dtype, x.device
        if self._cache_k is None:
            # cache_k shape: (batch_num, max_seq_len, num_kv_heads, head_dim)
            self._cache_k = torch.empty(
                (
                    self.args.max_batch_size,
                    self.args.max_seq_len,
                    self.n_kv_heads,
                    self.args.head_dim,
                ),
                dtype=dtype,
                device=device,
            )
        if self._cache_v is None:
            # cache_v shape: (batch_num, max_seq_len, num_kv_heads, head_dim)
            self._cache_v = torch.empty(
                (
                    self.args.max_batch_size,
                    self.args.max_seq_len,
                    self.n_kv_heads,
                    self.args.head_dim,
                ),
                dtype=dtype,
                device=device,
            )
        # Return key and value caches
        return self._cache_k, self._cache_v

    def forward(self, x, freqs_cis, positions, mask):
        # x shape: (batch_num, seq_len, embed_dim)
        bsz, seqlen, _ = x.shape
        # Get or initialize KV caches
        cache_k, cache_v = self.get_caches(x)

        # Project input to Q, K, V
        # Input x: (batch_num, seq_len, embed_dim)
        # Output xq: (batch_num, seq_len, num_heads * head_dim)
        # Output xk: (batch_num, seq_len, num_kv_heads * head_dim)
        # Output xv: (batch_num, seq_len, num_kv_heads * head_dim)
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # Reshape to separate heads dimension (Grouped Query Attention)
        # Input: (batch_num, seq_len, num_heads * head_dim) -> Output xq: (batch_num, seq_len, num_heads, head_dim)
        xq = xq.view(bsz, seqlen, self.n_heads, self.args.head_dim)
        # Input: (batch_num, seq_len, num_kv_heads * head_dim) -> Output xk: (batch_num, seq_len, num_kv_heads, head_dim)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.args.head_dim)
        # Input: (batch_num, seq_len, num_kv_heads * head_dim) -> Output xv: (batch_num, seq_len, num_kv_heads, head_dim)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.args.head_dim)

        # Apply RoPE to Q and K
        # Input xq: (batch_num, seq_len, num_heads,    head_dim) -> Output: same shape, rotated
        # Input xk: (batch_num, seq_len, num_kv_heads, head_dim) -> Output: same shape, rotated
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)

        # Compute rolling-buffer write positions (modulo max_seq_len)
        # The cache is a rotating buffer
        # scatter_pos before repeat: (1, seq_len, 1, 1)
        scatter_pos = (positions % self.args.max_seq_len)[None, :, None, None]
        # scatter_pos shape: (batch_num, seq_len, num_kv_heads, head_dim)
        scatter_pos = scatter_pos.repeat(bsz, 1, self.n_kv_heads, self.args.head_dim)

        # Write K/V into the KV cache at the correct positions
        # cache_k[:bsz] shape: (batch_num, max_seq_len, num_kv_heads, head_dim)
        cache_k[:bsz].scatter_(dim=1, index=scatter_pos, src=xk)
        # cache_v[:bsz] shape: (batch_num, max_seq_len, num_kv_heads, head_dim)
        cache_v[:bsz].scatter_(dim=1, index=scatter_pos, src=xv)

        # Expand KV heads to match num_heads (GQA)
        if positions.shape[0] > 1:
            # Prefill: use current K/V directly
            # Input: (batch_num, seq_len, num_kv_heads, head_dim) -> Output: (batch_num, seq_len, num_heads, head_dim)
            key, value = repeat_kv(xk, xv, self.repeats)
        # use cache if seq_len is 1 (decode step)
        else:
            # Decode: retrieve from full KV cache up to cur_pos
            cur_pos = int(positions[-1].item() + 1)
            # Input: (batch_num, cur_pos, num_kv_heads, head_dim) -> Output: (batch_num, cur_pos, num_heads, head_dim)
            key, value = repeat_kv(
                cache_k[:bsz, :cur_pos, ...],
                cache_v[:bsz, :cur_pos, ...],
                self.repeats,
            )

        # Transpose heads to front for batched matmul
        # Input: (batch_num, seq_len, num_heads, head_dim) -> Output query: (batch_num, num_heads, seq_len, head_dim)
        query = xq.transpose(1, 2)
        # Input: (batch_num, seq_len, num_heads, head_dim) -> Output key: (batch_num, num_heads, seq_len, head_dim)
        key = key.transpose(1, 2)
        # Input: (batch_num, seq_len, num_heads, head_dim) -> Output value: (batch_num, num_heads, seq_len, head_dim)
        value = value.transpose(1, 2)

        # Scaled dot-product attention scores
        # Q: (batch_num, num_heads, seq_len, head_dim) x K^T: (batch_num, num_heads, head_dim, seq_len)
        # Output scores: (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(query, key.transpose(2, 3)) * self.scale

        # Apply causal mask (additive, -inf for masked positions)
        # mask broadcast shape: (batch_num, num_heads, seq_len, seq_len)
        if mask is not None:
            scores += mask[None, None, ...]

        scores = scores.float()
        # Softmax over key positions
        # Input: (batch_num, num_heads, seq_len, seq_len) -> Output: same shape, normalized
        scores = nn.functional.softmax(scores, dim=-1).type_as(query)

        # Weighted aggregation of values
        # Input scores: (batch_num, num_heads, seq_len, seq_len), value: (batch_num, num_heads, seq_len, head_dim)
        # Output: (batch_num, num_heads, seq_len, head_dim)
        output = torch.matmul(scores, value)

        # Merge heads and project to embed_dim
        # Input: (batch_num, num_heads, seq_len, head_dim) -> (batch_num, seq_len, num_heads * head_dim) -> (batch_num, seq_len, embed_dim)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        # Input: (batch_num, seq_len, num_heads * head_dim) -> Output: (batch_num, seq_len, embed_dim)
        return self.wo(output)

In [ ]:
# Define the Sparse Mixture-of-Experts (MoE) Layer with top-k routing
class MoeLayer(nn.Module):
    def __init__(self, experts, gate, moe_args):
        super().__init__()
        # Assert that there is at least one expert
        assert len(experts) > 0

        # List of num_experts FeedForward networks
        self.experts = nn.ModuleList(experts)
        # Gate (router): linear layer projecting embed_dim -> num_experts
        self.gate = gate
        # Store MoE arguments
        self.args = moe_args

    def forward(self, inputs: torch.Tensor):
        # inputs shape: (batch_num, seq_len, embed_dim)

        # Flatten batch and sequence dimensions for token-level routing
        # Input: (batch_num, seq_len, embed_dim) -> Output inputs_squashed: (num_tokens, embed_dim)
        #   where num_tokens = batch_num * seq_len
        inputs_squashed = inputs.view(-1, inputs.shape[-1])

        # Compute routing logits for expert selection
        # Input: (num_tokens, embed_dim) -> Output gate_logits: (num_tokens, num_experts)
        gate_logits = self.gate(inputs_squashed)

        # Select top-k experts per token (sparse routing)
        # Input gate_logits: (num_tokens, num_experts)
        # Output weights:          (num_tokens, num_experts_per_tok)  -- raw top-k scores
        # Output selected_experts: (num_tokens, num_experts_per_tok)  -- expert indices
        weights, selected_experts = torch.topk(
            gate_logits, self.args.num_experts_per_tok)

        # Normalize routing weights via softmax over selected experts
        # Input: (num_tokens, num_experts_per_tok) -> Output weights: (num_tokens, num_experts_per_tok)
        weights = nn.functional.softmax(
            weights, dim=1, dtype=torch.float).type_as(inputs)

        # Initialize results accumulator tensor
        # results shape: (num_tokens, embed_dim)
        results = torch.zeros_like(inputs_squashed)

        # Dispatch tokens to each expert and accumulate weighted outputs
        for i, expert in enumerate(self.experts):
            # Find which tokens are routed to expert i
            # batch_idx shape:  (selected_m,)  -- token positions assigned to this expert
            # nth_expert shape: (selected_m,)  -- which of the top-k slots chose this expert
            batch_idx, nth_expert = torch.where(selected_experts == i)

            # Expert forward pass and weighted accumulation
            # expert(inputs_squashed[batch_idx]) shape: (selected_m, embed_dim)
            # weights[batch_idx, nth_expert, None] shape: (selected_m, 1)  -- broadcast scale
            # results[batch_idx] accumulated shape: (selected_m, embed_dim)
            results[batch_idx] += ( weights[batch_idx, nth_expert, None] *
                expert(inputs_squashed[batch_idx]) )

        # Reshape results back to (batch_num, seq_len, embed_dim)
        # Input: (num_tokens, embed_dim) -> Output: (batch_num, seq_len, embed_dim)
        return results.view_as(inputs)

In [ ]:
# Define a Transformer Block with Sparse MoE replacing the dense FFN
class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        # Get number of heads and embedding dimension
        self.n_heads = args.n_heads
        self.dim = args.dim

        # Multi-head attention (GQA, no sliding window)
        self.attention = Attention(args)

        # MoE layer: num_experts FeedForward networks with top-k gating
        self.feed_forward = MoeLayer(
            experts=[FeedForward(args=args) for _ in range(args.moe.num_experts)],
            gate=nn.Linear(args.dim, args.moe.num_experts, bias=False),
            moe_args=args.moe,
        )

        # Pre-attention RMSNorm
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        # Pre-FFN RMSNorm
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.args = args

    def forward(self, x, freqs_cis, positions, mask):
        # Input tensor shape: (batch_num, seq_len, embed_dim)

        # Pre-norm before attention
        # Input x: (batch_num, seq_len, embed_dim) -> norm_x: (batch_num, seq_len, embed_dim)
        norm_x = self.attention_norm(x)

        # Grouped-query self-attention
        # Input: (batch_num, seq_len, embed_dim) -> Output r: (batch_num, seq_len, embed_dim)
        r = self.attention.forward(norm_x, freqs_cis, positions, mask)

        # First residual connection
        # h shape: (batch_num, seq_len, embed_dim)
        h = x + r

        # Pre-norm before MoE FFN
        # Input h: (batch_num, seq_len, embed_dim) -> norm_h: (batch_num, seq_len, embed_dim)
        norm_h = self.ffn_norm(h)

        # Sparse MoE FeedForward (top-k expert routing)
        # Input: (batch_num, seq_len, embed_dim) -> Output r: (batch_num, seq_len, embed_dim)
        r = self.feed_forward.forward(norm_h)

        # Second residual connection
        # out shape: (batch_num, seq_len, embed_dim)
        out = h + r
        return out

In [ ]:
# Define the Mixtral Transformer with optional pipeline parallelism
class Transformer(nn.Module):
    def __init__(self, args, pipeline_rank=0, num_pipeline_ranks=1):
        super().__init__()
        self.args = args
        # Get vocabulary size, pipeline rank, and number of pipeline ranks
        self.vocab_size = args.vocab_size
        self.pipeline_rank = pipeline_rank
        self.num_pipeline_ranks = num_pipeline_ranks
        self._precomputed_freqs_cis = None

        # Rank-conditional module initialization
        self.tok_embeddings = None
        self.norm = None
        self.output = None

        # Token embedding only on first pipeline rank
        # tok_embeddings weight shape: (vocab_size, embed_dim)
        if pipeline_rank == 0:
            self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)

        # Norm + LM head only on last pipeline rank
        if pipeline_rank == num_pipeline_ranks - 1:
            self.norm = RMSNorm(args.dim, eps=args.norm_eps)
            # output: embed_dim -> vocab_size
            self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

        # Initialize all layers but slice off those not assigned to this rank
        layers = [TransformerBlock(args=args) for _ in range(args.n_layers)]
        # Calculate the number of layers per pipeline rank
        num_layers_per_rank = math.ceil(args.n_layers / self.num_pipeline_ranks)
        # Calculate the offset for the current pipeline rank
        offset = self.pipeline_rank * num_layers_per_rank
        # Calculate the end index for the current pipeline rank
        end = min(args.n_layers, offset + num_layers_per_rank)
        # Select the layers for the current pipeline rank
        self.layers = nn.ModuleDict({str(i): layers[i] for i in range(offset, end)})
        # Get the number of local layers for the current pipeline rank
        self.n_local_layers = len(self.layers)

    @property
    def dtype(self) -> torch.dtype:
        # Return the data type of the model parameters
        return next(self.parameters()).dtype

    @property
    def device(self) -> torch.device:
        # Return the device of the model parameters
        return next(self.parameters()).device

    @property
    def freqs_cis(self) -> torch.Tensor:
        # Lazily compute and cache RoPE frequencies; move to correct device if needed
        if self._precomputed_freqs_cis is None:
            # freqs_cis shape: (128_000, head_dim // 2) complex
            self._precomputed_freqs_cis = precompute_freqs_cis(
                head_dim=self.args.head_dim, end=128_000, theta=1000000.0)
        # Move freqs_cis to the correct device if necessary
        if self._precomputed_freqs_cis.device != self.device:
            self._precomputed_freqs_cis = self._precomputed_freqs_cis.to(device=self.device)
        # Return the precomputed frequencies
        return self._precomputed_freqs_cis

    def forward(self, input_ids, positions):
        # input_ids shape: (batch_num, seq_len)
        # positions shape: (seq_len,)

        # Slice RoPE frequencies for current positions
        # freqs_cis shape: (seq_len, head_dim // 2)
        freqs_cis = self.freqs_cis[positions]

        # Get batch size and sequence length
        (bsz, seqlen) = input_ids.shape
        # num_tokens = batch_num * seq_len (used in MoE dispatch)
        num_toks = bsz * seqlen

        # First pipeline rank: embed tokens
        if self.pipeline_rank == 0:
            # Embed input tokens
            # Input: (batch_num, seq_len) -> Output h: (batch_num, seq_len, embed_dim)
            h = self.tok_embeddings(input_ids)
        # Not the first pipeline rank: receive activations from previous rank
        else:
            # h shape: (batch_num, seq_len, embed_dim)
            h = torch.empty(bsz, seqlen, self.args.dim, device=self.device, dtype=self.dtype)
            torch.distributed.recv(h, src=self.pipeline_rank - 1)

        mask = None
        # Build causal attention mask for prefill (seq_len > 1)
        if input_ids.shape[1] > 1:
            seqlen = input_ids.shape[1]
            # Create a tensor filled with ones for masking
            # tensor shape: (seq_len, seq_len)
            tensor = torch.full(
                (seqlen, seqlen),
                dtype=h.dtype,
                fill_value=1,
                device=h.device,
            )
            # Lower-triangular causal mask converted to log scale
            # Input: (seq_len, seq_len) -> Output mask: (seq_len, seq_len)
            mask = torch.log(torch.tril(tensor, diagonal=0)).to(h.dtype)

        # Pass through this rank's transformer blocks
        # Input h: (batch_num, seq_len, embed_dim) -> Output h: (batch_num, seq_len, embed_dim)
        for layer in self.layers.values():
            # Apply transformer layer (with MoE FFN)
            # h shape: (batch_num, seq_len, embed_dim)
            h = layer(h, freqs_cis, positions, mask)

        # Pipeline communication: send to next rank or produce final logits
        if self.pipeline_rank < self.num_pipeline_ranks - 1:
            # Send intermediate hidden states to next pipeline rank
            torch.distributed.send(h, dst=self.pipeline_rank + 1)
            outs = torch.empty(*h.shape[:-1], self.vocab_size, device=h.device, dtype=h.dtype)
        else:
            # Final rank: apply norm then project to vocab logits
            # norm(h): (batch_num, seq_len, embed_dim)
            # Output outs: (batch_num, seq_len, vocab_size)
            outs = self.output(self.norm(h))

        # Broadcast final logits to all pipeline ranks
        if self.num_pipeline_ranks > 1:
            torch.distributed.broadcast(outs, src=self.num_pipeline_ranks - 1)

        # Output: (batch_num, seq_len, vocab_size)
        return outs.float()

    # Custom load_state_dict to selectively load weights per pipeline rank
    def load_state_dict(self, state_dict, *args, **kwargs):
        state_to_load = {}
        skipped = set([])
        for k, v in state_dict.items():
            # Load token embeddings for the first pipeline rank
            if k.startswith("tok_embeddings"):
                if self.pipeline_rank == 0:
                    state_to_load[k] = v
                else:
                    logging.debug(
                        "Skipping parameter %s at pipeline rank %d",
                        k,
                        self.pipeline_rank,
                    )
                    skipped.add(k)
            # Load norm and output layers for the last pipeline rank
            elif k.startswith("norm") or k.startswith("output"):
                if self.pipeline_rank == self.num_pipeline_ranks - 1:
                    state_to_load[k] = v
                else:
                    logging.debug(
                        "Skipping parameter %s at pipeline rank %d",
                        k,
                        self.pipeline_rank,
                    )
                    skipped.add(k)
            # Load transformer layers assigned to this pipeline rank
            elif k.startswith("layers"):
                layer_id = k.split(".")[1]
                if layer_id in self.layers:
                    state_to_load[k] = v
                else:
                    logging.debug(
                        "Skipping parameter %s at pipeline rank %d",
                        k,
                        self.pipeline_rank,
                    )
                    skipped.add(k)
            # Raise error for unexpected keys
            else:
                raise ValueError(f"Unexpected key {k}")
        # Assert that all keys are either loaded or skipped
        assert set(state_dict.keys()) == skipped.union(set(state_to_load.keys()))
        # Load the selected state dict
        super().load_state_dict(state_to_load, *args, **kwargs)

    @staticmethod
    def from_folder(
            folder, max_batch_size, max_seq_len, num_pipeline_ranks=1,
            device="cuda", dtype=torch.float16
        ):
        # Load model config from disk
        with open(folder / "params.json", "r") as f:
            model_args = ModelArgs.from_dict(json.load(f))
        # Set max batch size and max sequence length
        model_args.max_batch_size = max_batch_size
        model_args.max_seq_len = max_seq_len
        # Get the pipeline rank
        if num_pipeline_ranks > 1:
            pipeline_rank = torch.distributed.get_rank()
        else:
            pipeline_rank = 0
        # Instantiate on meta device to avoid allocating memory before weight loading
        with torch.device("meta"):
            model = Transformer(
                model_args,
                pipeline_rank=pipeline_rank,
                num_pipeline_ranks=num_pipeline_ranks
            )
        # Load and assign pretrained weights
        loaded = torch.load(str(folder / "consolidated.00.pth"), mmap=True)
        model.load_state_dict(loaded, assign=True)
        return model.to(device=device, dtype=dtype)

In [ ]:
# Instantiate Mixtral MoE Transformer and run a test forward pass
# Input: (batch_num=3, min_prompt_len)
model = Transformer(args).to("cuda", dtype=torch.float32)

# Forward pass through the full MoE model
# Input: (batch_num, min_prompt_len) -> Output logits: (batch_num, min_prompt_len, vocab_size)
logits = model.forward(input_tokens[:, :min_prompt_len], positions)

# Compute log-probabilities over the vocabulary
# Input: (batch_num, min_prompt_len, vocab_size) -> Output logprobs: (batch_num, min_prompt_len, vocab_size)
logprobs = nn.functional.log_softmax(logits, dim=-1)
# print(logits.size(), logprobs.size())

torch.Size([12]) torch.Size([12])
torch.Size([12]) torch.Size([12])
torch.Size([11]) torch.Size([11])
torch.Size([7]) torch.Size([7])
